# Transfer learning and fine-tuning with CNNs

Recognise 5 kinds of flowers from a few hundred photos by reusing CNNs pretrained on ImageNet.
Run the cells in order. The first run downloads the dataset and the pretrained weights. See `README.md` for details and things to try.

## 1. Pretrained CNNs as frozen feature extractors

Each pretrained CNN, without its 1000-class output layer, turns every photo into a feature vector. It is only run, never trained. A new output layer for our 5 classes is trained on those vectors.

In [ ]:
# Transfer learning in one cell: ResNet50 vs MobileNetV2 vs EfficientNetB0 on 5 kinds of flowers
# (if needed, install once with:  %pip install tensorflow matplotlib)
import os, sys
# Only matters for the Windows GPU environment in the README (lets TensorFlow find the NVIDIA files); harmless elsewhere
os.environ["PATH"] = os.path.join(sys.prefix, "Library", "bin") + os.pathsep + os.environ["PATH"]
import time
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

print("Running on:", "GPU" if tf.config.list_physical_devices("GPU") else "CPU only (no GPU found)")

# 1. Sample dataset: 3,670 flower photos in 5 folders (daisy, dandelion, roses, sunflowers, tulips).
#    Downloaded once (about 230 MB) and kept in ~/.keras/datasets
URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
archive = keras.utils.get_file(origin=URL, extract=True)
data_dir = next(Path(archive).parent.rglob("roses")).parent   # the folder that holds the 5 class folders

train, val = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="both", seed=42, image_size=(224, 224), batch_size=32)
classes = train.class_names
train, val = train.take(15).cache(), val.take(8).cache()      # keep it small: 480 training and 256 validation photos
y_train = np.concatenate([y for _, y in train])               # labels (this also loads the photos into memory)
y_val = np.concatenate([y for _, y in val])

# 2. Three CNNs pretrained on ImageNet; each expects its pixels prepared its own way (preprocess_input)
pretrained = {
    "ResNet50":       (keras.applications.ResNet50,       keras.applications.resnet50.preprocess_input),
    "MobileNetV2":    (keras.applications.MobileNetV2,    keras.applications.mobilenet_v2.preprocess_input),
    "EfficientNetB0": (keras.applications.EfficientNetB0, keras.applications.efficientnet.preprocess_input),
}

results = {}
for name, (Net, preprocess) in pretrained.items():
    print(f"{name}: loading the pretrained model and computing features ...", flush=True)
    # Pretrained CNN without its 1000-class output layer. It stays frozen: we only run it, never train it.
    base = Net(include_top=False, weights="imagenet", input_shape=(224, 224, 3), pooling="avg")

    # Run every photo through it ONCE and keep the feature vector it produces (the slow part)
    t0 = time.perf_counter()
    X_train = base.predict(train.map(lambda x, y: preprocess(x)), verbose=0)
    X_val = base.predict(val.map(lambda x, y: preprocess(x)), verbose=0)
    t_features = time.perf_counter() - t0

    # Train only a new output layer for our 5 classes on those features (takes seconds)
    head = keras.Sequential([keras.Input(X_train.shape[1:]),
                             keras.layers.Dense(len(classes), activation="softmax")])
    head.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    hist = head.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=20, verbose=0)
    results[name] = hist.history["val_accuracy"][-1]
    print(f"{name:15s} frozen weights: {base.count_params():>10,} | trained weights: {head.count_params():>6,} | "
          f"features {t_features:4.0f} s + training {time.perf_counter() - t0 - t_features:3.0f} s | "
          f"validation accuracy: {results[name]:.1%}")

# 3. Compare
plt.bar(list(results), list(results.values()), color=["tab:blue", "tab:orange", "tab:green"])
plt.ylim(0, 1)
plt.ylabel("validation accuracy")
plt.title(f"Transfer learning with {len(y_train)} training photos")
plt.show()

## 2. Freezing a pretrained network

`base.trainable = False` freezes the network inside a model we train. The cell counts the trainable weights before and after, trains the model, and checks which weights changed.

In [ ]:
# Freezing a pretrained network, step by step (run after the cell above: it reuses its train/val photos)
import numpy as np
from tensorflow import keras

base = keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(224, 224, 3), pooling="avg")
preprocess = keras.applications.mobilenet_v2.preprocess_input


def n_weights(weights):
    return sum(int(np.prod(w.shape)) for w in weights)


print(f"Before freezing: {n_weights(base.trainable_weights):>9,} trainable weights in MobileNetV2")
base.trainable = False                       # <-- THIS LINE FREEZES THE PRETRAINED NETWORK
print(f"After freezing:  {n_weights(base.trainable_weights):>9,} trainable weights in MobileNetV2\n")

# New model = frozen pretrained network + a new output layer for our 5 classes
inputs = keras.Input((224, 224, 3))
features = base(preprocess(inputs), training=False)   # training=False: BatchNorm layers keep their ImageNet statistics
new_layer = keras.layers.Dense(len(classes), activation="softmax")
model = keras.Model(inputs, new_layer(features))
model.summary()                              # see the last lines: trainable vs non-trainable params

# Proof: train the whole model, then check which weights changed
base_before = [w.copy() for w in base.get_weights()]
new_before = new_layer.get_weights()[0].copy()
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train, validation_data=val, epochs=3, verbose=2)
base_same = all(np.array_equal(a, b) for a, b in zip(base_before, base.get_weights()))
new_changed = not np.array_equal(new_before, new_layer.get_weights()[0])
print(f"\nPretrained MobileNetV2 weights unchanged after training: {base_same}")
print(f"New output layer weights changed after training:        {new_changed}")

# Next cell: fine-tuning unfreezes the last layers again

## 3. Fine-tuning

Unfreeze the last 30 layers and train them together with the output layer, with a learning rate 100 times smaller so the pretrained weights are adjusted, not overwritten.

In [ ]:
# Fine-tuning: unfreeze the last layers of the pretrained network and train them a little
# (run after the cell above: it continues training the same model)
base.trainable = True
for layer in base.layers[:-30]:                  # keep everything except the last 30 layers frozen
    layer.trainable = False
print(f"Trainable weights now: {n_weights(model.trainable_weights):,} (last 30 layers of MobileNetV2 + output layer)")

# Recompile after changing what is trainable. The learning rate is 100 times smaller than before,
# so the pretrained weights are adjusted, not overwritten. BatchNorm layers stay in inference mode
# because the model calls base(..., training=False).
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
acc_before = model.evaluate(val, verbose=0)[1]
hist = model.fit(train, validation_data=val, epochs=5, verbose=2)
acc_after = model.evaluate(val, verbose=0)[1]
print(f"\nValidation accuracy: {acc_before:.1%} before fine-tuning, {acc_after:.1%} after")

plt.plot(hist.history["accuracy"], "o-", label="training")
plt.plot(hist.history["val_accuracy"], "o-", label="validation")
plt.xlabel("fine-tuning epoch")
plt.ylabel("accuracy")
plt.title("Fine-tuning MobileNetV2: watch the gap between the two curves")
plt.legend()
plt.show()